# Title

Description

## Imports

In [26]:
import pandas as pd

import matplotlib.pyplot as plt

import sys

import os

import torch

import json
import torch

from transformers import AutoTokenizer, AutoModel

from algorithm import Algorithm

from pathlib import Path

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import tensorflow as tf

## 1. Load dataframes

In [2]:
ai_train = pd.read_csv("../Dataframes/df_ai/df_ai_train.csv")
ai_val = pd.read_csv("../Dataframes/df_ai/df_ai_val.csv")
ai_test = pd.read_csv("../Dataframes/df_ai/df_ai_test.csv")

plagiarism_train = pd.read_csv("../Dataframes/df_plagiarism/df_plagiarism_train.csv")
plagiarism_val = pd.read_csv("../Dataframes/df_plagiarism/df_plagiarism_val.csv")
plagiarism_test = pd.read_csv("../Dataframes/df_plagiarism/df_plagiarism_test.csv")

print("------AI shapes------")
print("Train: ", ai_train.shape)
print("Validation: ", ai_val.shape)
print("Test: ", ai_test.shape)
print("------Plagiarism shapes------")
print("Train: ", plagiarism_train.shape)
print("Validation: ", plagiarism_val.shape)
print("Test: ", plagiarism_test.shape)

ai_train.head()

------AI shapes------
Train:  (24000, 20)
Validation:  (3000, 20)
Test:  (3000, 20)
------Plagiarism shapes------
Train:  (24000, 56)
Validation:  (3000, 56)
Test:  (3000, 56)


,code,label,approx_tokens,comment_density,avg_line_length,line_length_variance,blank_line_ratio,num_classes,num_methods,num_if,num_for,num_while,num_switch,o_complexity,max_depth,total_nodes,num_literals,num_ids,unique_ids,id_diversity
0,private Set<Integer> perNodeRelease(final C th...,0,101,0.0,61.882353,1268.339100,0.055556,0.0,0.0,0.0,0.0,0.0,0.0,1.0,6.0,0.0,0.0,63.0,30.0,0.476190
1,@Override\r\n public AuthenticationStatus f...,0,23,0.0,34.777778,702.395062,0.100000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,19.0,15.0,0.789474
2,public void callWorkListenerWithError(WorkCont...,1,28,0.0,49.500000,2072.250000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,5.0,0.0,0.0,7.0,7.0,1.000000
3,public void setSubscription(Subscription s) {\...,1,14,0.0,33.250000,500.187500,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,6.0,5.0,0.833333
4,public boolean getDialogContentInset(int theme...,1,20,0.0,44.666667,1461.222222,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,21.0,16.0,0.761905


## 2. Separate attribites and labels

In [3]:
X_ai_train = ai_train.drop("label", axis=1)
y_ai_train = ai_train["label"]

X_ai_val = ai_val.drop("label", axis=1)
y_ai_val = ai_val["label"]

X_ai_test = ai_test.drop("label", axis=1)
y_ai_test = ai_test["label"]

X_plagiarism_train = plagiarism_train.drop("label", axis=1)
y_plagiarism_train = plagiarism_train["label"]

X_plagiarism_val = plagiarism_val.drop("label", axis=1)
y_plagiarism_val = plagiarism_val["label"]

X_plagiarism_test = plagiarism_test.drop("label", axis=1)
y_plagiarism_test = plagiarism_test["label"]


print("X_ai_train:", X_ai_train.shape)
print("y_ai_train:", y_ai_train.shape)
print("X_ai_val:", X_ai_val.shape)
print("y_ai_val:", y_ai_val.shape)
print("X_ai_test:", X_ai_test.shape)
print("y_ai_test:", y_ai_test.shape)

print("X_plagiarism_train:", X_plagiarism_train.shape)
print("y_plagiarism_train:", y_plagiarism_train.shape)
print("X_plagiarism_val:", X_plagiarism_val.shape)
print("y_plagiarism_val:", y_plagiarism_val.shape)
print("X_plagiarism_test:", X_plagiarism_test.shape)
print("y_plagiarism_test:", y_plagiarism_test.shape)

print("\nTraining class distribution:")
print(y_ai_train.value_counts().sort_index())
print("\nValidation class distribution:")
print(y_ai_val.value_counts().sort_index())
print("\nTesting class distribution:")
print(y_ai_test.value_counts().sort_index())

print("\nTraining class distribution:")
print(y_plagiarism_train.value_counts().sort_index())
print("\nValidation class distribution:")
print(y_plagiarism_val.value_counts().sort_index())
print("\nTesting class distribution:")
print(y_plagiarism_test.value_counts().sort_index())


X_ai_train: (24000, 19)
y_ai_train: (24000,)
X_ai_val: (3000, 19)
y_ai_val: (3000,)
X_ai_test: (3000, 19)
y_ai_test: (3000,)
X_plagiarism_train: (24000, 55)
y_plagiarism_train: (24000,)
X_plagiarism_val: (3000, 55)
y_plagiarism_val: (3000,)
X_plagiarism_test: (3000, 55)
y_plagiarism_test: (3000,)

Training class distribution:
label
0    12000
1    12000
Name: count, dtype: int64

Validation class distribution:
label
0    1500
1    1500
Name: count, dtype: int64

Testing class distribution:
label
0    1500
1    1500
Name: count, dtype: int64

Training class distribution:
label
0    12000
1    12000
Name: count, dtype: int64

Validation class distribution:
label
0    1500
1    1500
Name: count, dtype: int64

Testing class distribution:
label
0    1500
1    1500
Name: count, dtype: int64


## 3. Import model

In [24]:
sys.path.append(os.path.abspath("../Dataset/conplag_version_2/"))

from scripts.codebert import create_model

tokenizer, model = create_model()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 33134.84it/s]


## 4. Compile modelo

In [28]:
# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
#     loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
#     metrics=["accuracy"]
# )